# Emerging Tech Lab - Solvent-Screen Opentrons Protocol

## Solvent-screen protocol setup

In [ ]:
from solvent_screen_opentrons_helpers import (
    log_step,
    build_solvent_screen_plate_map,
    validate_volume_for_p300,
    dispense_to_wells_with_tip_changes,
    set_robot_speeds,
    log_absolute_time_tick,
    summarise_solvent_screen_records,
)

import opentrons.execute
protocol = opentrons.execute.get_protocol_api("2.19")

# Loading labware
plate_48 = protocol.load_labware(
    "greenaway_48_wellplate_3750ul",
    location=2,
)
plate_8 = protocol.load_labware(
    "greenaway_8_wellplate_20000ul",
    location=3,
)
pip_rack = protocol.load_labware(
    "opentrons_96_tiprack_300ul",
    location=1,
)

# Load pipette
pip_300 = protocol.load_instrument(
    "p300_single_gen2",
    "left",  # change to "right" if needed
    tip_racks=[pip_rack],
)

# -----------------------------
# User setup: liquid handling
# -----------------------------

ASPIRATE_RATE = 70
STANDARD_DISPENSE_RATE = 70
SLOW_DISPENSE_RATE = 10  # used for dropwise dialdehyde addition
AIR_GAP_VOLUME = 15
MAX_DISPENSE = 200
PRE_WET_CYCLES = 3
PRE_WET_VOLUME = 180
TIP_CHANGE_INTERVAL = 3
TRANSFER_MODE = "fast"  # "fast" = one tip per reagent/solvent source; "accurate" = change tips regularly

pip_300.flow_rate.aspirate = ASPIRATE_RATE
pip_300.flow_rate.dispense = STANDARD_DISPENSE_RATE

# Speed up robot movement while keeping liquid-handling flow rates controlled.
# Rim-touching during pre-wetting is slowed separately inside the helper function.
set_robot_speeds(
    protocol=protocol,
    pipette=pip_300,
    pipette_default_speed=400,
    max_head_speed=400,
)

# -----------------------------
# User setup: source locations
# -----------------------------

# 8-well source plate layout:
# A1-A4: diamine stocks in different solvents
# B1-B4: dialdehyde stocks in matching solvents
SOLVENT_CONDITIONS = {
    "chloroform": {
        "diamine_source": plate_8["A1"],
        "dialdehyde_source": plate_8["B1"],
        "row": "A",
    },
    "methanol": {
        "diamine_source": plate_8["A2"],
        "dialdehyde_source": plate_8["B2"],
        "row": "B",
    },
    "chloroform_methanol_1to1": {
        "diamine_source": plate_8["A3"],
        "dialdehyde_source": plate_8["B3"],
        "row": "C",
    },
    "hexane": {
        "diamine_source": plate_8["A4"],
        "dialdehyde_source": plate_8["B4"],
        "row": "D",
    },
}

# Select the solvent condition(s) for this run.
# Each condition uses one row and three replicate wells, e.g. A1-A3.
CONDITIONS_TO_RUN = ["chloroform", "methanol", "chloroform_methanol_1to1", "hexane"]

if len(CONDITIONS_TO_RUN) == 0:
    raise ValueError("Select at least one solvent condition to run.")

N_REPLICATES = 3
START_COLUMN = 1

# -----------------------------
# User setup: dispense volumes
# -----------------------------

# Enter the calculated volume for each stock solution per reaction vial.
# These volumes are applied to every replicate well in this run.
volume_of_diamine = 1000  # uL per well
volume_of_dialdehyde = 200  # uL per well

# -----------------------------
# Build and validate the plate map
# -----------------------------

validate_volume_for_p300(volume_of_diamine, "Diamine", protocol=protocol)
validate_volume_for_p300(volume_of_dialdehyde, "Dialdehyde", protocol=protocol)

condition_plate_map = build_solvent_screen_plate_map(
    solvent_conditions=SOLVENT_CONDITIONS,
    conditions_to_run=CONDITIONS_TO_RUN,
    n_replicates=N_REPLICATES,
    start_column=START_COLUMN,
    protocol=protocol,
)

## Automated solvent-screen setup

In [ ]:
# -----------------------------
# Automated solvent-screen setup
# Correct addition order:
#   1. diamine solution
#   2. dialdehyde solution, slow/dropwise
# -----------------------------

# Store dispense records for later inspection.
diamine_dispense_records = {}
dialdehyde_dispense_records = {}

# Add diamine solution to all replicate wells for each solvent condition.
# This does not start the imine reaction until dialdehyde is added.
for condition_name in CONDITIONS_TO_RUN:
    condition = condition_plate_map[condition_name]
    diamine_dispense_records[condition_name] = dispense_to_wells_with_tip_changes(
        pipette=pip_300,
        protocol=protocol,
        plate=plate_48,
        source_well=condition["diamine_source"],
        target_well_names=condition["target_wells"],
        total_volume=volume_of_diamine,
        dispense_rate=STANDARD_DISPENSE_RATE,
        reagent_name=f"diamine stock ({condition_name})",
        tip_change_interval=TIP_CHANGE_INTERVAL,
        transfer_mode=TRANSFER_MODE,
        max_dispense=MAX_DISPENSE,
        air_gap_volume=AIR_GAP_VOLUME,
        pre_wet_cycles=PRE_WET_CYCLES,
        pre_wet_volume=PRE_WET_VOLUME,
        log_each_dispense_time=False,
        log_absolute_time=False,
    )

# Add dialdehyde solution slowly/dropwise to start the reaction for each solvent condition.
for condition_name in CONDITIONS_TO_RUN:
    condition = condition_plate_map[condition_name]
    log_absolute_time_tick(protocol, f"Starting dialdehyde addition for solvent-screen condition {condition_name}.")

    dialdehyde_dispense_records[condition_name] = dispense_to_wells_with_tip_changes(
        pipette=pip_300,
        protocol=protocol,
        plate=plate_48,
        source_well=condition["dialdehyde_source"],
        target_well_names=condition["target_wells"],
        total_volume=volume_of_dialdehyde,
        dispense_rate=SLOW_DISPENSE_RATE,
        reagent_name=f"dialdehyde stock ({condition_name})",
        tip_change_interval=TIP_CHANGE_INTERVAL,
        transfer_mode=TRANSFER_MODE,
        max_dispense=MAX_DISPENSE,
        air_gap_volume=AIR_GAP_VOLUME,
        pre_wet_cycles=PRE_WET_CYCLES,
        pre_wet_volume=PRE_WET_VOLUME,
        log_each_dispense_time=True,
        log_absolute_time=True,
    )

    log_absolute_time_tick(protocol, f"Finished dialdehyde addition for solvent-screen condition {condition_name}.")

# Reset dispense rate and summarise the run.
pip_300.flow_rate.dispense = STANDARD_DISPENSE_RATE

log_step(protocol, "Solvent-screen summary by condition:")
summarise_solvent_screen_records(
    dispense_records_by_condition=dialdehyde_dispense_records,
    protocol=protocol,
)

protocol.home()